In [ ]:
# Step 1: Install required libraries (latest versions to avoid conflicts)
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
# Step 2: Write the custom model class to a REAL .py file on disk.
# This is required because newer versions of transformers try to open and read
# the source file of any class inheriting from PreTrainedModel.
# In a Jupyter notebook, __main__ has no real file, so we must write one.

model_code = '''
import torch
import torch.nn as nn
from transformers import BertModel, BertPreTrainedModel

class BertForMultiLabelClassification(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(config.hidden_size, self.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.float())

        return {"loss": loss, "logits": logits}
'''

with open('toxic_model.py', 'w') as f:
    f.write(model_code)

print("Model file 'toxic_model.py' written to disk successfully!")

In [ ]:
# Step 3: Import libraries and the model class from the .py file
import os
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

from datasets import Dataset

from transformers import (
    BertModel,
    BertTokenizer,
    BertConfig,
    Trainer,
    TrainingArguments
)

# Import custom model from the .py file we just created
from toxic_model import BertForMultiLabelClassification

# Disable WandB logging
os.environ["WANDB_DISABLED"] = "true"

print("All imports successful!")

In [ ]:
# Step 4: Load Dataset
if os.path.exists("/content/train.csv"):
    dataset_path = "/content/train.csv"
else:
    dataset_path = "train.csv"

print(f"Loading dataset from: {dataset_path}")

df = pd.read_csv(
    dataset_path,
    engine="python",
    on_bad_lines="skip"
)

print("Dataset loaded successfully!")

In [ ]:
# Step 5: Define Labels
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

In [ ]:
# Step 6: Split Data into Train and Validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["comment_text"].astype(str).tolist(),
    df[labels].values,
    test_size=0.1,
    random_state=42
)

In [ ]:
# Step 7: Create Datasets
train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "labels": val_labels.tolist()
})

In [ ]:
# Step 8: Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Step 9: Configure and Initialize Model with Pretrained Weights
config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_labels=len(labels)
)

# Initialize custom model
model = BertForMultiLabelClassification(config)

# Load pretrained BERT weights into the model's bert encoder
pretrained_bert = BertModel.from_pretrained("bert-base-uncased")
model.bert.load_state_dict(pretrained_bert.state_dict())

print("Model loaded successfully!")

In [ ]:
# Step 10: Training Configuration
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
# Step 11: Metrics Computation
def compute_metrics(eval_pred):
    logits, labels_arr = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.4).astype(int)
    labels_arr = labels_arr.astype(int)

    acc = accuracy_score(labels_arr, preds)
    f1 = f1_score(labels_arr, preds, average="macro", zero_division=0)
    precision = precision_score(labels_arr, preds, average="macro", zero_division=0)
    recall = recall_score(labels_arr, preds, average="macro", zero_division=0)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# Step 12: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# Step 13: Train Model
trainer.train()

In [ ]:
# Step 14: Save Trained Model
output_model_dir = "./results/best_model"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Best model saved at: {output_model_dir}")

In [ ]:
# Step 15: Create ZIP Archive
print("Creating ZIP file...")
shutil.make_archive("best_model", "zip", output_model_dir)
print("ZIP file created successfully!")

In [ ]:
# Step 16: Download the Zip File
try:
    from google.colab import files
    files.download("best_model.zip")
except ImportError:
    print("Not running in Colab. The zip file 'best_model.zip' is saved in your current directory.")